In [ ]:
import dash
from dash import dcc, html, Input, Output
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
#  DATA LOADING
# ─────────────────────────────────────────────
df = pd.read_csv("../data/cleaned_layoff_dataset.csv")

country_mapping = {
    "US": "United States",
    "UK": "United Kingdom",
    "UAE": "United Arab Emirates",
    "South Korea": "Korea, South",
    "Russia": "Russian Federation",
}
df["country_full"] = df["country"].replace(country_mapping).fillna(df["country"])

df["date"]          = pd.to_datetime(df["date"], errors="coerce")
df["year"]          = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
df["month"]         = pd.to_numeric(df["month"], errors="coerce").astype("Int64")
df["layoff_count"]  = pd.to_numeric(df["layoff_count"], errors="coerce")
df["pct_workforce"] = pd.to_numeric(df["pct_workforce"], errors="coerce")
df["raised_mm"]     = pd.to_numeric(df["raised_mm"], errors="coerce")
df["quarter"]       = df["date"].dt.to_period("Q").astype(str)
df["year_quarter"]  = df["quarter"].copy()

years      = sorted(df["year"].dropna().unique().tolist())
industries = sorted(df["industry"].dropna().unique().tolist())

# ─────────────────────────────────────────────
#  DESIGN TOKENS
# ─────────────────────────────────────────────
BG       = "#ffffff"
CARD     = "#ffffff"
CARD2    = "#f7f8fa"
BORDER   = "#e4e7ed"
TEXT     = "#1a1d23"
MUTED    = "#6b7280"
GRID     = "#e9ecf0"

C1       = "#2563eb"
C2       = "#f97316"
C3       = "#10b981"
C4       = "#8b5cf6"
C5       = "#ef4444"
C_LINE   = "#1d4ed8"
COLORWAY = [C1, C2, C3, C4, C5, "#06b6d4", "#ec4899"]
FONT     = "'Inter', 'Segoe UI', sans-serif"

PLOT_BASE = dict(
    paper_bgcolor=CARD,
    plot_bgcolor=CARD,
    font=dict(family=FONT, color=TEXT, size=12),
    margin=dict(l=48, r=24, t=40, b=48),
    xaxis=dict(
        gridcolor=GRID, zerolinecolor=GRID,
        tickfont=dict(size=11, color=TEXT),
        title_font=dict(size=12, color=TEXT),
        linecolor=BORDER,
    ),
    yaxis=dict(
        gridcolor=GRID, zerolinecolor=GRID,
        tickfont=dict(size=11, color=TEXT),
        title_font=dict(size=12, color=TEXT),
        linecolor=BORDER,
    ),
    legend=dict(
        bgcolor=CARD, bordercolor=BORDER, borderwidth=1,
        font=dict(size=11, color=TEXT),
    ),
    colorway=COLORWAY,
)

# ─────────────────────────────────────────────
#  HELPER COMPONENTS
# ─────────────────────────────────────────────
def section_title(text):
    return html.Div(text, style={
        "fontSize": "11px", "fontWeight": "600", "letterSpacing": "1.5px",
        "textTransform": "uppercase", "color": C1,
        "marginBottom": "12px", "marginTop": "4px",
    })

def card(children, style_extra=None):
    s = {
        "background": CARD,
        "border": f"1px solid {BORDER}",
        "borderRadius": "10px",
        "padding": "20px",
        "boxShadow": "0 1px 4px rgba(0,0,0,0.06)",
    }
    if style_extra:
        s.update(style_extra)
    return html.Div(children, style=s)

def kpi(label, value_id, color=C1):
    return html.Div([
        html.Div(label, style={
            "fontSize": "11px", "color": MUTED, "fontWeight": "500",
            "letterSpacing": "0.5px", "textTransform": "uppercase", "marginBottom": "6px",
        }),
        html.Div(id=value_id, style={
            "fontSize": "26px", "fontWeight": "700", "color": color, "lineHeight": "1",
        }),
    ], style={
        "background": CARD2, "border": f"1px solid {BORDER}",
        "borderTop": f"3px solid {color}",
        "borderRadius": "8px", "padding": "16px 20px",
        "flex": "1", "minWidth": "130px",
    })

# ─────────────────────────────────────────────
#  LAYOUT
# ─────────────────────────────────────────────
app2 = dash.Dash(__name__, suppress_callback_exceptions=True)
app2.title = "Layoff Analytics Dashboard v2"

app2.layout = html.Div(style={
    "background": "#f0f2f5", "minHeight": "100vh",
    "fontFamily": FONT, "padding": "28px 36px",
}, children=[

    # ── HEADER ──────────────────────────────────
    html.Div([
        html.H1("Layoff Analytics Dashboard", style={
            "margin": "0", "fontSize": "22px", "fontWeight": "700", "color": TEXT,
        }),
        html.Div("Global tech workforce reduction intelligence", style={
            "fontSize": "13px", "color": MUTED, "marginTop": "4px",
        }),
    ], style={"marginBottom": "24px"}),

    # ── GLOBAL FILTERS ───────────────────────────
    card([
        html.Div([
            html.Div([
                html.Div("Year Range", style={"fontSize": "12px", "color": MUTED, "marginBottom": "8px", "fontWeight": "500"}),
                dcc.RangeSlider(
                    id="year-slider-v2",
                    min=years[0], max=years[-1], step=1,
                    marks={y: {"label": str(y), "style": {"color": MUTED, "fontSize": "11px"}} for y in years},
                    value=[years[0], years[-1]],
                    tooltip={"always_visible": False},
                ),
            ], style={"flex": "2", "minWidth": "260px"}),

            html.Div([
                html.Div("Industry", style={"fontSize": "12px", "color": MUTED, "marginBottom": "8px", "fontWeight": "500"}),
                dcc.Dropdown(
                    id="industry-filter-v2",
                    options=[{"label": i, "value": i} for i in industries],
                    multi=True, placeholder="All industries",
                ),
            ], style={"flex": "3", "minWidth": "240px"}),

            html.Div([
                html.Div("Company Type", style={"fontSize": "12px", "color": MUTED, "marginBottom": "8px", "fontWeight": "500"}),
                dcc.RadioItems(
                    id="ai-toggle-v2",
                    options=[
                        {"label": "  All",     "value": "all"},
                        {"label": "  AI only", "value": "ai"},
                        {"label": "  Non-AI",  "value": "nonai"},
                    ],
                    value="all",
                    labelStyle={"display": "block", "fontSize": "13px", "color": TEXT, "marginBottom": "6px"},
                    inputStyle={"accentColor": C1, "marginRight": "6px"},
                ),
            ], style={"flex": "1", "minWidth": "130px"}),
        ], style={"display": "flex", "gap": "36px", "flexWrap": "wrap", "alignItems": "flex-start"}),
    ], style_extra={"marginBottom": "20px"}),

    # ── KPI ROW ──────────────────────────────────
    html.Div([
        kpi("Total Layoffs",            "kpi-total-v2",     C1),
        kpi("Companies Affected",       "kpi-companies-v2", C2),
        kpi("Avg % Workforce Affected", "kpi-avgpct-v2",    C3),
        kpi("Complete Shutdowns",       "kpi-shutdown-v2",  C5),
        kpi("Countries",                "kpi-countries-v2", C4),
    ], style={"display": "flex", "gap": "14px", "flexWrap": "wrap", "marginBottom": "20px"}),

    # ── ROW 1: Treemap + Quarterly Trend ─────────
    html.Div([
        html.Div([
            section_title("Goal 1 — Share by Industry"),
            card([
                html.Div([
                    html.Div([
                        html.Div("Layoff Distribution Treemap", style={"fontSize": "14px", "fontWeight": "600", "color": TEXT, "marginBottom": "2px"}),
                        html.Div("Area proportional to total layoffs", style={"fontSize": "11px", "color": MUTED}),
                    ]),
                    html.Div([
                        dcc.Dropdown(
                            id="industry-topn-v2",
                            options=[
                                {"label": "Top 5",  "value": 5},
                                {"label": "Top 10", "value": 10},
                                {"label": "All",    "value": "all"},
                            ],
                            value=5, clearable=False, searchable=False,
                            style={"width": "120px", "fontSize": "12px"}
                        )
                    ]),
                ], style={"display": "flex", "justifyContent": "space-between", "alignItems": "center", "marginBottom": "10px"}),
                dcc.Graph(id="industry-treemap-v2", config={"displayModeBar": False}, style={"height": "340px"}),
            ]),
        ], style={"flex": "1", "minWidth": "300px"}),

        html.Div([
            section_title("Goal 2 — Temporal Trends"),
            card([
                html.Div("Quarterly Layoff Trend", style={"fontSize": "14px", "fontWeight": "600", "color": TEXT, "marginBottom": "2px"}),
                html.Div("Bars = quarterly total · Line = 2-quarter rolling avg", style={"fontSize": "11px", "color": MUTED, "marginBottom": "10px"}),
                dcc.Graph(id="quarterly-trend-v2", config={"displayModeBar": False}, style={"height": "340px"}),
            ]),
        ], style={"flex": "1", "minWidth": "300px"}),
    ], style={"display": "flex", "gap": "18px", "flexWrap": "wrap", "marginBottom": "18px"}),

    # ── ROW 2: AI Resilience + Bubble Chart ──────
    html.Div([
        html.Div([
            section_title("Goal 3 — AI vs Non-AI Resilience"),
            card([
                html.Div("Avg % Workforce Affected by Layoffs", style={"fontSize": "14px", "fontWeight": "600", "color": TEXT, "marginBottom": "2px"}),
                html.Div("Grouped by Year · AI vs Non-AI companies", style={"fontSize": "11px", "color": MUTED, "marginBottom": "10px"}),
                dcc.Graph(id="ai-violin-v2", config={"displayModeBar": False}, style={"height": "340px"}),
            ]),
        ], style={"flex": "1", "minWidth": "300px"}),

        html.Div([
            section_title("Goal 5 — Funding Stage Vulnerability"),
            card([
                html.Div("Avg Layoffs vs Event Frequency by Stage", style={"fontSize": "14px", "fontWeight": "600", "color": TEXT, "marginBottom": "2px"}),
                html.Div("Bubble size = total layoffs · color = stage", style={"fontSize": "11px", "color": MUTED, "marginBottom": "10px"}),
                dcc.Graph(id="stage-bubble-v2", config={"displayModeBar": False}, style={"height": "340px"}),
            ]),
        ], style={"flex": "1", "minWidth": "300px"}),
    ], style={"display": "flex", "gap": "18px", "flexWrap": "wrap", "marginBottom": "18px"}),

    # ── ROW 3: Geography ─────────────────────────
    html.Div([
        section_title("Goal 4 — Geographic Distribution"),
        card([
            html.Div([
                html.Div([
                    html.Div("Global Layoff Choropleth", style={"fontSize": "14px", "fontWeight": "600", "color": TEXT}),
                    html.Div("Click & drag to rotate globe · hover for detail", style={"fontSize": "11px", "color": MUTED}),
                ], style={"marginBottom": "12px"}),
                html.Div([
                    dcc.Graph(id="choropleth-globe-v2", config={"displayModeBar": False},
                              style={"height": "420px", "flex": "1", "minWidth": "340px"}),
                    html.Div([
                        html.Div("Top 10 Cities by Layoff Count", style={"fontSize": "14px", "fontWeight": "600", "color": TEXT, "marginBottom": "2px"}),
                        html.Div("Total layoffs per metro area", style={"fontSize": "11px", "color": MUTED, "marginBottom": "10px"}),
                        dcc.Graph(id="top-cities-v2", config={"displayModeBar": False}, style={"height": "380px"}),
                    ], style={"flex": "1", "minWidth": "220px"}),
                ], style={"display": "flex", "gap": "20px", "flexWrap": "wrap"}),
            ]),
        ]),
    ], style={"marginBottom": "18px"}),

    # ── FOOTER ────────────────────────────────────
    html.Div("Data: cleaned_layoff_dataset.csv  ·  Built with Dash + Plotly", style={
        "textAlign": "center", "color": MUTED, "fontSize": "11px",
        "paddingTop": "14px", "borderTop": f"1px solid {BORDER}",
    }),
])

# ─────────────────────────────────────────────
#  FILTER HELPER
# ─────────────────────────────────────────────
def filter_df_v2(year_range, industries_sel, ai_toggle):
    d = df[(df["year"] >= year_range[0]) & (df["year"] <= year_range[1])].copy()
    if industries_sel:
        d = d[d["industry"].isin(industries_sel)]
    if ai_toggle == "ai":
        d = d[d["is_ai_company"] == True]
    elif ai_toggle == "nonai":
        d = d[d["is_ai_company"] == False]
    return d

# ─────────────────────────────────────────────
#  CALLBACKS
# ─────────────────────────────────────────────

# KPIs
@app2.callback(
    Output("kpi-total-v2",     "children"),
    Output("kpi-companies-v2", "children"),
    Output("kpi-avgpct-v2",    "children"),
    Output("kpi-shutdown-v2",  "children"),
    Output("kpi-countries-v2", "children"),
    Input("year-slider-v2",    "value"),
    Input("industry-filter-v2","value"),
    Input("ai-toggle-v2",      "value"),
)
def update_kpis_v2(yr, ind, ai):
    d         = filter_df_v2(yr, ind, ai)
    total     = int(d["layoff_count"].sum())
    companies = d["company"].nunique()
    avg_pct   = round(d["pct_workforce"].mean(), 1)
    shutdown  = round((d["pct_workforce"] == 100).sum() / max(len(d), 1) * 100, 1)
    countries = d["country"].nunique()
    return f"{total:,}", f"{companies:,}", f"{avg_pct}%", f"{shutdown}%", str(countries)

# Treemap
@app2.callback(
    Output("industry-treemap-v2", "figure"),
    Input("year-slider-v2",       "value"),
    Input("industry-filter-v2",   "value"),
    Input("ai-toggle-v2",         "value"),
    Input("industry-topn-v2",     "value"),
)
def update_treemap_v2(yr, ind, ai, topn):
    d = filter_df_v2(yr, ind, ai).copy()
    d["layoff_count"] = d["layoff_count"].fillna(0)
    grp = d.groupby(["industry", "company"]).agg(total_layoffs=("layoff_count", "sum")).reset_index()
    grp = grp[grp["total_layoffs"] > 0]
    industry_totals = grp.groupby("industry")["total_layoffs"].sum().sort_values(ascending=False)
    if topn != "all":
        top_industries = industry_totals.head(int(topn)).index
        grp = grp[grp["industry"].isin(top_industries)]
    fig = px.treemap(grp, path=["industry", "company"], values="total_layoffs",
                     color="total_layoffs", color_continuous_scale="Blues")
    fig.update_traces(
        texttemplate="<b>%{label}</b><br>%{value:,}",
        hovertemplate="<b>%{label}</b><br>Layoffs: %{value:,}<extra></extra>",
        marker=dict(line=dict(width=1, color="#ffffff"))
    )
    layout = {**PLOT_BASE}
    layout.update(margin=dict(t=10, b=10, l=10, r=10), coloraxis_showscale=False)
    fig.update_layout(**layout)
    return fig

# Quarterly Trend
@app2.callback(
    Output("quarterly-trend-v2", "figure"),
    Input("year-slider-v2",      "value"),
    Input("industry-filter-v2",  "value"),
    Input("ai-toggle-v2",        "value"),
)
def update_quarterly_v2(yr, ind, ai):
    d   = filter_df_v2(yr, ind, ai)
    grp = d.groupby("quarter")["layoff_count"].sum().reset_index().sort_values("quarter")
    grp["rolling"] = grp["layoff_count"].rolling(2, min_periods=1).mean()
    fig = go.Figure()
    fig.add_trace(go.Bar(x=grp["quarter"], y=grp["layoff_count"],
                         name="Quarterly total", marker_color="#93c5fd",
                         hovertemplate="%{x}<br>Layoffs: %{y:,}<extra></extra>"))
    fig.add_trace(go.Scatter(x=grp["quarter"], y=grp["rolling"],
                             mode="lines+markers", line=dict(color=C1, width=2.5),
                             marker=dict(size=5, color=C1), name="2Q rolling avg"))
    layout = {**PLOT_BASE}
    layout.update(xaxis_tickangle=-45, xaxis_title="Quarter", yaxis_title="Layoffs")
    fig.update_layout(**layout)
    return fig

# AI Resilience Bar
@app2.callback(
    Output("ai-violin-v2",      "figure"),
    Input("year-slider-v2",     "value"),
    Input("industry-filter-v2", "value"),
)
def update_ai_bar_v2(yr, ind):
    d = filter_df_v2(yr, ind, "all").dropna(subset=["pct_workforce"])
    d["label"] = d["is_ai_company"].map({True: "AI", False: "Non-AI"})
    grp = d.groupby(["year", "label"]).agg(avg_pct=("pct_workforce", "mean")).reset_index()
    fig = px.bar(grp, x="year", y="avg_pct", color="label", barmode="group",
                 color_discrete_map={"AI": C2, "Non-AI": C1},
                 labels={"avg_pct": "Avg % Workforce Affected", "year": "Year", "label": "Type"})
    fig.update_traces(hovertemplate="<b>%{customdata[0]} Companies</b><br>Year: %{x}<br>Avg: %{y:.1f}%<extra></extra>",
                      customdata=grp[["label"]].values)
    layout = {**PLOT_BASE}
    layout.update(xaxis=dict(type="category", gridcolor=GRID,
                             tickfont=dict(size=11, color=TEXT),
                             title_font=dict(size=12, color=TEXT)),
                  yaxis_title="Avg % Workforce Affected",
                  legend=dict(title="", x=0.85, y=0.95,
                              bgcolor="rgba(255,255,255,0.9)",
                              bordercolor=BORDER, borderwidth=1))
    fig.update_layout(**layout)
    return fig

# ── BUBBLE CHART (V1 version — x=avg_layoffs, y=events, size=Total) ─────────
@app2.callback(
    Output("stage-bubble-v2",   "figure"),
    Input("year-slider-v2",     "value"),
    Input("industry-filter-v2", "value"),
    Input("ai-toggle-v2",       "value"),
)
def update_bubble_v2(yr, ind, ai):
    d = filter_df_v2(yr, ind, ai)
    stage_df = (
        d.groupby("stage").agg(
            Total_layoffs  =("layoff_count", "sum"),
            average_layoffs=("layoff_count", "mean"),
            layoff_events  =("layoff_count", "count")
        ).reset_index()
    )
    fig = px.scatter(
        stage_df,
        x="average_layoffs",
        y="layoff_events",
        size="Total_layoffs",
        color="stage",
        title="",
        labels={
            "average_layoffs": "Average Layoffs per Event",
            "layoff_events"  : "Number of Layoff Events",
            "Total_layoffs"  : "Total Layoffs",
            "stage"          : "Funding Stage"
        },
        hover_name="stage",
        size_max=70,
        text="stage",
        template="plotly_white"
    )
    fig.update_traces(textposition="top center", mode="markers+text",
                      textfont=dict(size=10))
    layout = {**PLOT_BASE}
    layout.update(
        xaxis_title="Average Layoffs per Event",
        yaxis_title="Number of Layoff Events",
        showlegend=False,
        margin=dict(l=48, r=24, t=20, b=48)
    )
    fig.update_layout(**layout)
    return fig

# Choropleth Globe
@app2.callback(
    Output("choropleth-globe-v2", "figure"),
    Input("year-slider-v2",       "value"),
    Input("industry-filter-v2",   "value"),
    Input("ai-toggle-v2",         "value"),
)
def update_globe_v2(yr, ind, ai):
    d   = filter_df_v2(yr, ind, ai)
    grp = d.groupby("country_full")["layoff_count"].sum().reset_index()
    fig = go.Figure(go.Choropleth(
        locations=grp["country_full"], locationmode="country names",
        z=grp["layoff_count"],
        colorscale=[[0, "#ccccff"], [0.4, "#3366ff"], [1, "#000080"]],
        marker_line_color="#ffffff", marker_line_width=0.5,
        colorbar=dict(
            title=dict(text="Layoffs", font=dict(color=TEXT, family=FONT, size=12)),
            tickfont=dict(color=TEXT, family=FONT, size=10),
            bgcolor="rgba(255,255,255,0.85)",
            outlinewidth=1, outlinecolor=BORDER, len=0.6,
        ),
        hovertemplate="<b>%{location}</b><br>Layoffs: %{z:,}<extra></extra>",
    ))
    fig.update_geos(projection_type="orthographic",
                    showland=True, landcolor="#ffffff",
                    showocean=True, oceancolor="#ecf1f6",
                    showframe=False, showcountries=True,
                    countrycolor="#311305", showcoastlines=True,
                    coastlinecolor="#6B2005", bgcolor=CARD)
    fig.update_layout(paper_bgcolor=CARD, geo_bgcolor=CARD,
                      margin=dict(l=0, r=0, t=0, b=0),
                      font=dict(family=FONT, color=TEXT))
    return fig

# Top Cities
@app2.callback(
    Output("top-cities-v2",     "figure"),
    Input("year-slider-v2",     "value"),
    Input("industry-filter-v2", "value"),
    Input("ai-toggle-v2",       "value"),
)
def update_cities_v2(yr, ind, ai):
    d   = filter_df_v2(yr, ind, ai)
    grp = d.groupby("location")["layoff_count"].sum().nlargest(10).reset_index().sort_values("layoff_count")
    fig = go.Figure(go.Bar(
        y=grp["location"], x=grp["layoff_count"], orientation="h",
        marker=dict(color=grp["layoff_count"],
                    colorscale=[[0, "#ccccff"], [0.4, "#3366ff"], [1, "#000080"]],
                    showscale=False),
        text=grp["layoff_count"].apply(lambda v: f"{int(v):,}"),
        textposition="outside", textfont=dict(size=10, color=TEXT),
        hovertemplate="<b>%{y}</b><br>Layoffs: %{x:,}<extra></extra>",
    ))
    layout = {**PLOT_BASE}
    layout.update(xaxis_title="Total layoffs",
                  margin=dict(l=12, r=60, t=20, b=36),
                  yaxis=dict(tickfont=dict(size=10, color=TEXT), gridcolor=GRID))
    fig.update_layout(**layout)
    return fig

# ─────────────────────────────────────────────
#  RUN
# ─────────────────────────────────────────────
if __name__ == "__main__":
    app2.run(debug=True, port=8051)